<a href="https://colab.research.google.com/github/sevastopaul2047/document-instruction-safety/blob/main/notebooks/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lexsi real Tiny Aya pilot
Use a GPU runtime (T4 or better), preferably Python 3.12. Accept the model's Hugging Face access conditions and add a read token as the Colab secret `HF_TOKEN`. This notebook runs 27 matched cases (81 responses), with 4-bit NF4 loading. It has not yet been executed on a GPU. No fine-tuning.


In [3]:
import os, sys, subprocess
from pathlib import Path
assert sys.version_info >= (3, 12), "Select a Python 3.12+ runtime"
REPO = "https://github.com/sevastopaul2047/document-instruction-safety.git"
REF = "main"  # use main after the PR is merged, or a reproducible commit
ROOT = Path("/content/document-instruction-safety")
if ROOT.exists():
    raise RuntimeError("Repository path already exists; use a fresh runtime or review the existing checkout")
subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
subprocess.run(["git", "checkout", REF], cwd=ROOT, check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

In [4]:
from google.colab import userdata
# Never print the token. Child processes inherit it; no token is saved to the repo.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# Run the access/GPU check in a fresh process after pip installs. Downloads only tokenizer/config.
preflight = """
import torch
from huggingface_hub import model_info
from transformers import AutoConfig, AutoTokenizer
from lexsi_pilot.inference import render_chat
assert torch.cuda.is_available(), 'Choose a GPU runtime'
model_id = 'CohereLabs/tiny-aya-global'
sha = model_info(model_id).sha
config = AutoConfig.from_pretrained(model_id, revision=sha, trust_remote_code=False)
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=sha, trust_remote_code=False)
_, mode = render_chat(tokenizer, [{'role':'system','content':'Use facts.'},{'role':'user','content':'Hello'}])
print({'gpu':torch.cuda.get_device_name(), 'revision':sha, 'model_type':config.model_type, 'role_mode':mode})
"""
subprocess.run([sys.executable, "-c", preflight], check=True)


CompletedProcess(args=['/usr/bin/python3', '-c', "\nimport torch\nfrom huggingface_hub import model_info\nfrom transformers import AutoConfig, AutoTokenizer\nfrom lexsi_pilot.inference import render_chat\nassert torch.cuda.is_available(), 'Choose a GPU runtime'\nmodel_id = 'CohereLabs/tiny-aya-global'\nsha = model_info(model_id).sha\nconfig = AutoConfig.from_pretrained(model_id, revision=sha, trust_remote_code=False)\ntokenizer = AutoTokenizer.from_pretrained(model_id, revision=sha, trust_remote_code=False)\n_, mode = render_chat(tokenizer, [{'role':'system','content':'Use facts.'},{'role':'user','content':'Hello'}])\nprint({'gpu':torch.cuda.get_device_name(), 'revision':sha, 'model_type':config.model_type, 'role_mode':mode})\n"], returncode=0)

A 401/403 means the token/account still needs model access. Resolve that on Hugging Face, then rerun the credential cell. The frozen PIArena selection is already present; live dataset access is optional. If role mode is `fold`, the defense will be a user-prefix instruction and must be reported as such.


In [ ]:
from datetime import datetime, timezone

RUN = Path("results") / ("debug-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

result = subprocess.run(
    [sys.executable, "scripts/run_pilot.py",
     "--model", "transformers",
     "--data", "piarena",
     "--output", str(RUN)],
    capture_output=True,
    text=True
)

print("OUTPUT:", result.stdout[-2000:])
print("ERROR:", result.stderr[-8000:])

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(RUN), "zip", RUN)
files.download(archive)


Review `manual_review.csv`, raw outputs, unknown counts, input errors, and per-condition denominators before interpreting results. The mock tests are synthetic validation; only this GPU run produces Tiny Aya outputs. Preserve unreviewed files when adding human judgments.
